# Live metadata demo — YouTube Shorts

Этот ноутбук работает **с локальным backend репозитория**, поэтому API key хранится только в `.env` у сервера.

Цель — показать metadata-first отбор без обязательного скачивания MP4:
1. получить реальные публичные metadata;
2. построить ranking;
3. повторить поиск позже и получить фактический `Δ views/hour`;
4. экспортировать выбранного кандидата в JSON-контракт для ComfyUI.


In [ ]:
import httpx
import pandas as pd
import matplotlib.pyplot as plt

BASE_URL = "http://127.0.0.1:8000"
QUERY = "AI tools"

health = httpx.get(f"{BASE_URL}/health", timeout=10).json()
health

In [ ]:
payload = {
    "query": QUERY,
    "max_age_hours": 168,
    "min_views": 1000,
    "min_views_per_hour": 0,
    "min_like_rate": 0,
    "max_duration_seconds": 180,
    "limit": 25,
}
response = httpx.post(f"{BASE_URL}/api/search", json=payload, timeout=30)
response.raise_for_status()
data = response.json()

df = pd.DataFrame(data["videos"])
cols = [
    "title", "channel_title", "views", "views_per_hour",
    "growth_views_per_hour", "like_rate", "comment_rate",
    "breakout_ratio", "popularity_score"
]
df[cols].head(15)

In [ ]:
plot_df = df.head(15).sort_values("popularity_score")
plt.figure(figsize=(10, 6))
plt.barh(plot_df["title"].str.slice(0, 45), plot_df["popularity_score"])
plt.xlabel("Metadata popularity score")
plt.title(f"Top candidates: {QUERY}")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 6))
plt.scatter(df["views_per_hour"], df["like_rate"] * 100)
plt.xlabel("Views/hour")
plt.ylabel("Like rate, %")
plt.title("Velocity vs engagement — metadata only")
plt.tight_layout()
plt.show()

## Фактическая динамика

При первом запуске `growth_views_per_hour` будет пустым: backend сохраняет первый snapshot.

Для расчёта динамики повторите запрос к `/api/search` через некоторое время. Backend сравнит текущие views с предыдущим snapshot и вернёт скорость роста **между двумя наблюдениями**, не скачивая видео.


In [ ]:
growth = df.dropna(subset=["growth_views_per_hour"]).copy()
if growth.empty:
    print("Повторных snapshot'ов пока нет. Выполните тот же поиск позже.")
else:
    growth = growth.sort_values("growth_views_per_hour", ascending=False).head(15)
    plt.figure(figsize=(10, 6))
    plt.barh(growth["title"].str.slice(0, 45)[::-1], growth["growth_views_per_hour"][::-1])
    plt.xlabel("Δ views/hour")
    plt.title("Real growth velocity between snapshots")
    plt.tight_layout()
    plt.show()

## ComfyUI handoff

На этом этапе MP4 не обязателен. Для выбранного кандидата backend формирует стабильный JSON manifest. Поля `comfyui_inputs` можно сопоставить с input-полями любого API-format workflow через отдельный adapter.


In [ ]:
top = data["videos"][0]
manifest_response = httpx.post(
    f"{BASE_URL}/api/comfy/manifest",
    json={"video": top, "local_video_path": None},
    timeout=10,
)
manifest_response.raise_for_status()
manifest = manifest_response.json()["manifest"]
manifest

In [ ]:
from pathlib import Path
Path("../exports").mkdir(exist_ok=True)
out = Path("../exports") / f"{top['video_id']}.comfy-manifest.json"
out.write_text(__import__("json").dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print(out.resolve())